# TikTok Recipe Intelligence - Component Smoke Tests

Run this notebook from the repository root or from the `notebooks/` directory. It is designed to test each brick independently without hiding failures behind Airflow.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import requests

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'docker' / 'docker-compose.yml').exists():
    REPO_ROOT = Path.cwd().parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(REPO_ROOT)

## 1. Environment variables

In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / '.env', override=True)
required = [
    'SNOWFLAKE_USER',
    'SNOWFLAKE_PASSWORD',
    'SNOWFLAKE_ACCOUNT',
    'SNOWFLAKE_WAREHOUSE',
    'SNOWFLAKE_DB',
    'OPENROUTER_API_KEY',
]
missing = [name for name in required if not os.getenv(name)]
print('Missing:', missing)
assert not missing, f'Missing env vars: {missing}'

## 2. Docker Compose configuration

In [ ]:
result = subprocess.run(
    ['docker', 'compose', '-f', 'docker/docker-compose.yml', 'config', '--services'],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0

## 3. Snowflake connectivity and table visibility

In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DB'),
    role=os.getenv('SNOWFLAKE_ROLE'),
)
with conn.cursor() as cur:
    cur.execute('select current_database(), current_warehouse(), current_role()')
    print(cur.fetchone())
    for schema, table in [
        ('BRONZE', 'BRONZE_TIKTOK_RECIPES'),
        ('SILVER', 'SILVER_TIKTOK_RECIPES'),
        ('GOLD', 'GOLD_API_RECIPE_CATALOG'),
    ]:
        cur.execute(f'select count(*) from {schema}.{table}')
        print(schema, table, cur.fetchone()[0])
conn.close()

## 4. LLM parser without network call

In [ ]:
from scripts.enrich_silver import normalize_llm_enrichment

examples = [
    {'lang': 'en', 'is_veg': False, 'cuisine': 'italian', 'ingredient': 'fettuccine'},
    [{'lang': 'fr', 'is_veg': True, 'cuisine': 'french', 'ingredient': 'mushroom'}],
    {'recipe_language': 'es', 'is_vegetarian': False, 'cuisine_style': 'mexican', 'main_ingredient': 'pollo'},
]
for item in examples:
    parsed = normalize_llm_enrichment(item)
    print(parsed.model_dump())

## 5. Bronze CSV ingestion dry command

In [ ]:
csv_files = sorted((REPO_ROOT / 'data' / 'raw').glob('*.csv'))
print([p.name for p in csv_files])
print('Run manually when ready: python -m scripts.load_bronze --input-dir data/raw')

## 6. dbt command wiring

In [ ]:
result = subprocess.run(
    [sys.executable, 'run_dbt.py', 'debug'],
    text=True,
    capture_output=True,
    timeout=120,
)
print(result.stdout[-2000:])
print(result.stderr[-2000:])
assert result.returncode == 0

## 7. API health

In [ ]:
response = requests.get('http://localhost:18000/health', timeout=15)
print(response.status_code)
print(response.text[:1000])
assert response.ok

## 8. Streamlit availability

In [ ]:
response = requests.get('http://localhost:18501', timeout=15)
print(response.status_code)
print(response.text[:200])
assert response.ok

## 9. Kafka host/container sanity

In [ ]:
print('Host bootstrap:', 'localhost:19092')
print('Container bootstrap:', os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'kafka:29092'))
print('Producer test from host: python -m scripts.kafka_producer --bootstrap-server localhost:19092 --topic recipes_raw')

## 10. Spark analytics command

In [ ]:
cmd = [
    'docker', 'exec', 'docker-spark-analytics-1', 'bash', '-c',
    '/opt/spark/bin/spark-submit --conf spark.jars.ivy=/tmp/.ivy2 '
    '--packages net.snowflake:snowflake-jdbc:3.15.1,net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4 '
    '/app/scripts/spark_recipe_analytics.py'
]
print(' '.join(cmd))
print('Uncomment the next lines to run Spark from this notebook.')
# result = subprocess.run(cmd, text=True, capture_output=True, timeout=600)
# print(result.stdout[-4000:])
# print(result.stderr[-4000:])
# assert result.returncode == 0